# Qualitative Mathematical Modelling in Python

Author: Jayden Hyman

Version: 0.2.21

Email: j.hyman@uq.edu.au

📝 Getting started:

1. Install the QMM package: run the cell `%pip install qmm-core`, then comment out the line `%pip install qmm-core` with `#` at the start of the line, or simply delete the cell. Then restart the kernel from the menu bar.
2. Create a signed digraph using the web application: [Open in browser](https://d2x70551if0frn.cloudfront.net/).
3. Run the Python package and dependencies cell.
4. Import the signed digraph file (`.json`) using the `import_digraph` function (e.g., `import_digraph("folder/model.json")`). You can also use the `list_to_digraph` to create a signed digraph from a signed adjacency matrix (e.g., `list_to_digraph("[[-1,-1,0],[1,-1,-1],[0,1,-1]]")`).
5. Run each cell by pressing the keyboard shortcut "shift+enter" or the play icon (when hovering over a cell). 

The lake-mesocosm model from Hulot et al. (2000) is used as an example in this notebook.

Reference: Hulot, F.D., Lacroix, G., Lescher-Moutoué, F., Loreau, M., 2000. Functional diversity governs ecosystem response to nutrient enrichment. Nature 405, 340–344. https://doi.org/10.1038/35012591

In [ ]:
%pip install -e C:\Users\uqjhyman\Documents\qmm

In [18]:
#@title Python package and dependencies
from qmm import *

## Model structure

In [19]:
G = import_digraph("test/snowshoe-hare-extended.json")
G = define_state_space(G)
create_matrix(G, form='signed')

Matrix([
[-1,  1,  1,  0],
[-1, -1,  1,  0],
[ 0, -1, -1,  1],
[ 0,  0,  0, -1]])

## Stability analysis

In [20]:
sign_stability(G)

,Test,Definition,Result
0,Condition i,No positive self-effects,True
1,Condition ii,At least one node is self-regulating,True
2,Condition iii,The product of any pairwise interaction is non-positive,True
3,Condition iv,No cycles greater than length two,False
4,Condition v,Non-zero determinant (all nodes have at least one incoming and outgoing link),True
5,Colour test,Fails Jeffries' colour test,True
6,Sign stable,Satisfies necessary and sufficient conditions for sign stability,False


In [21]:
feedback_metrics(G)

,Feedback level,Net,Absolute,Positive,Negative,Weighted
0,0,-1,1,0,1,-1
1,1,-4,4,0,4,-1
2,2,-8,8,0,8,-1
3,3,-7,9,1,8,-7/9
4,4,-2,4,1,3,-1/2


In [22]:
determinants_metrics(G)

,Hurwitz determinant,Net,Absolute,Weighted
0,0,1,1,1
1,1,4,4,1
2,2,25,41,25/41
3,3,143,433,143/433
4,4,286,1732,143/866


In [23]:
conditional_stability(G)

,Test,Definition,Result
0,Weighted feedback,Maximum weighted feedback (level 4),-0.50
1,Weighted determinant,n-1 weighted determinant at level,0.33
2,Ratio to model-c system,Ratio to a 'model-c' type system,3.6
3,Model class,Class of the model based on conditional stability metrics,Class I


In [24]:
simulation_stability(G, n_sim=10000)

,Test,Definition,Result
0,Stable matrices,Proportion where all eigenvalues have negative real parts,85.35%
1,Unstable matrices,Proportion where one or more eigenvalues have positive real parts,14.65%
2,Hurwitz criterion i,Proportion where polynomial coefficients are not all of the same sign,14.65%
3,Hurwitz criterion ii,Proportion where Hurwitz determinants are not all positive,1.81%
4,Hurwitz criterion i only,Proportion where only Hurwitz criterion i fails,12.84%
5,Hurwitz criterion ii only,Proportion where only Hurwitz criterion ii fails,0.00%


## Press perturbation analysis

In [25]:
adjoint_matrix(G, form='signed')

Matrix([
[ 2,  0, 2, 2],
[-1,  1, 0, 0],
[ 1, -1, 2, 2],
[ 0,  0, 0, 2]])

In [26]:
absolute_feedback_matrix(G)

Matrix([
[2, 2, 2, 2],
[1, 1, 2, 2],
[1, 1, 2, 2],
[0, 0, 0, 4]])

In [27]:
weighted_predictions_matrix(G, as_nan=False, as_abs=True).evalf(2)

Matrix([
[1.0,   0, 1.0, 1.0],
[1.0, 1.0,   0,   0],
[1.0, 1.0, 1.0, 1.0],
[1.0, 1.0, 1.0, 0.5]])

In [28]:
sign_determinacy_matrix(G, method='average', as_nan=False, as_abs=True).evalf(2)

Matrix([
[1.0, 0.5, 1.0,  1.0],
[1.0, 1.0, 0.5,  0.5],
[1.0, 1.0, 1.0,  1.0],
[1.0, 1.0, 1.0, 0.86]])

In [29]:
numerical_simulations(G, n_sim=10000, dist="uniform", as_nan=False, as_abs=True).evalf(2)

Matrix([
[1.0, 0.59,  1.0,  1.0],
[1.0,  1.0, 0.59, 0.59],
[1.0,  1.0,  1.0,  1.0],
[  0,    0,    0,  1.0]])

## Qualitative predictions

In [30]:
index, columns = get_nodes(G, 'state'), get_nodes(G, 'state')
print("Qualitative predictions (prediction weights):")
table_of_predictions(weighted_predictions_matrix(G), t1=0.5, t2=1, index=index, columns=columns)

Qualitative predictions (prediction weights):


,1,2,3,4
1,+,?,+,+
2,−,+,?,?
3,+,−,+,+
4,0,0,0,(+)


In [31]:
print("Qualitative predictions (sign determinacy):")
table_of_predictions(sign_determinacy_matrix(G), t1=0.8, t2=1, index=index, columns=columns)

Qualitative predictions (sign determinacy):


,1,2,3,4
1,+,?,+,+
2,−,+,?,?
3,+,−,+,+
4,0,0,0,(+)


In [32]:
print("Qualitative predictions (numerical simulations):")
table_of_predictions(numerical_simulations(G), t1=0.8, t2=1, index=index, columns=columns)

Qualitative predictions (numerical simulations):


,1,2,3,4
1,+,?,+,+
2,−,+,?,?
3,+,−,+,+
4,0,0,0,+


## Extensions (experimental)

### Structural sensitivity

In [33]:
net_structural_sensitivity(G, level=None)

Matrix([
[-2, -1,  1,  0],
[ 0, -1, -1,  0],
[ 0,  0, -2,  0],
[ 0,  0,  0, -2]])

In [34]:
absolute_structural_sensitivity(G, level=None)

Matrix([
[2, 1, 1, 0],
[2, 1, 1, 0],
[0, 2, 2, 0],
[0, 0, 0, 4]])

In [35]:
weighted_structural_sensitivity(G, level=None).evalf(2)

Matrix([
[-1.0, -1.0,  1.0,  nan],
[   0, -1.0, -1.0,  nan],
[ nan,    0, -1.0,  nan],
[ nan,  nan,  nan, -0.5]])

### Change in life expectancy

In [36]:
birth_matrix(G, form='signed')

Matrix([
[0, 1, 1, 0],
[0, 0, 1, 0],
[0, 0, 0, 1],
[0, 0, 0, 0]])

In [37]:
death_matrix(G, form='signed')

Matrix([
[1, 0, 0, 0],
[1, 1, 0, 0],
[0, 1, 1, 0],
[0, 0, 0, 1]])

In [38]:
net_life_expectancy_change(G, type='birth')

Matrix([
[-2,  0, -2, -2],
[-1, -1, -2, -2],
[ 0,  0, -2, -2],
[ 0,  0,  0, -2]])

In [39]:
net_life_expectancy_change(G, type='death')

Matrix([
[ 0, 0, -2, -2],
[-1, 1, -2, -2],
[ 0, 0,  0, -2],
[ 0, 0,  0,  0]])

In [40]:
absolute_life_expectancy_change(G, type='birth')

Matrix([
[2, 2, 2, 2],
[1, 3, 2, 2],
[0, 0, 4, 4],
[0, 0, 0, 4]])

In [41]:
absolute_life_expectancy_change(G, type='death')

Matrix([
[2, 2, 2, 2],
[1, 1, 2, 2],
[0, 0, 0, 4],
[0, 0, 0, 0]])

In [42]:
weighted_predictions_life_expectancy(G, type='birth', as_nan=False, as_abs=True).evalf(2)

Matrix([
[1.0,    0, 1.0, 1.0],
[1.0, 0.33, 1.0, 1.0],
[1.0,  1.0, 0.5, 0.5],
[1.0,  1.0, 1.0, 0.5]])

In [43]:
weighted_predictions_life_expectancy(G, type='death', as_nan=False, as_abs=True).evalf(2)

Matrix([
[  0,   0, 1.0, 1.0],
[1.0, 1.0, 1.0, 1.0],
[1.0, 1.0, 1.0, 0.5],
[1.0, 1.0, 1.0, 1.0]])

In [44]:
bpred = table_of_predictions(weighted_predictions_life_expectancy(G, type='birth'), t1=0.5, t2=1)
dpred = table_of_predictions(weighted_predictions_life_expectancy(G, type='death'), t1=0.5, t2=1)
delta_le = compare_predictions(bpred, dpred)
delta_le.columns = get_nodes(G, 'state')
delta_le.index = get_nodes(G, 'state')
delta_le

,1,2,3,4
1,"−, ?",?,−,−
2,−,"?, +",−,−
3,0,0,"(−), 0",(−)
4,0,0,0,"(−), 0"


### Causal pathways

Specify source and target node id for selected analyses:

In [45]:
cycles_table(G)

,Length,Cycle,Sign
0,1,1 ⊸ 1,−
1,1,2 ⊸ 2,−
2,1,3 ⊸ 3,−
3,1,4 ⊸ 4,−
4,2,1 ⊸ 2 → 1,−
5,2,3 → 2 ⊸ 3,−
6,3,1 ⊸ 2 ⊸ 3 → 1,+


In [46]:
source='3'
target='1'
get_paths(G, source=source, target=target, form='signed')

Matrix([
[1],
[1]])

In [47]:
complementary_feedback(G, source=source, target=target, form='signed')

Matrix([
[-1],
[-1]])

In [48]:
system_paths(G, source=source, target=target, form='signed')

Matrix([
[1],
[1]])

In [49]:
weighted_paths(G, source=source, target=target)

Matrix([
[1],
[1]])

In [50]:
paths_table(G, source=source, target=target)

,Length,Path,Sign
0,1,3 → 1,+
1,2,3 → 2 → 1,+


In [51]:
get_paths(G, source=source, target=target, form="symbolic")

Matrix([
[a_1,2*a_2,3],
[      a_1,3]])

In [52]:
get_paths(G, source=source, target=target, form="signed")

Matrix([
[1],
[1]])

In [53]:
get_paths(G, source=source, target=target, form="binary")

Matrix([
[1],
[1]])

In [54]:
complementary_feedback(G, source=source, target=target, form="symbolic")

Matrix([
[      -a_4,4],
[-a_2,2*a_4,4]])

In [55]:
complementary_feedback(G, source=source, target=target, form="signed")

Matrix([
[-1],
[-1]])

In [56]:
complementary_feedback(G, source=source, target=target, form="binary")

Matrix([
[1],
[1]])

In [57]:
system_paths(G, source=source, target=target, form="symbolic")

Matrix([
[a_1,2*a_2,3*a_4,4],
[a_1,3*a_2,2*a_4,4]])

In [58]:
system_paths(G, source=source, target=target, form="signed")

Matrix([
[1],
[1]])

In [59]:
system_paths(G, source=source, target=target, form="binary")

Matrix([
[1],
[1]])

In [60]:
weighted_paths(G, source=source, target=target).evalf(2)

Matrix([
[1.0],
[1.0]])

In [61]:
path_metrics(G, source=source, target=target)

,Length,Path,Path sign,Complementary subsystem,Net feedback,Absolute feedback,Positive feedback,Negative feedback,Weighted feedback,Weighted path
0,1,"3, 1",+,"2, 4",-1,1,0,1,-1,1
1,2,"3, 2, 1",+,4,-1,1,0,1,-1,1


### State space representation

In [62]:
create_matrix(G, form='signed', matrix_type='A')

Matrix([
[-1,  1,  1,  0],
[-1, -1,  1,  0],
[ 0, -1, -1,  1],
[ 0,  0,  0, -1]])

In [63]:
create_matrix(G, form='signed', matrix_type='B')

Matrix([
[ 1,  0],
[-1,  0],
[ 0, -1],
[ 0,  0]])

In [64]:
create_matrix(G, form='signed', matrix_type='C')

Matrix([
[0, -1, 1, 0],
[0,  1, 0, 0]])

In [65]:
create_matrix(G, form='signed', matrix_type='D')

Matrix([
[0, 0],
[0, 1]])

In [66]:
net_effects(G)

Matrix([
[ 2,  0, 2, 2,  2, -2],
[-1,  1, 0, 0, -2,  0],
[ 1, -1, 2, 2,  2, -2],
[ 0,  0, 0, 2,  0,  0],
[ 2, -2, 2, 2,  4, -2],
[-1,  1, 0, 0, -2,  1]])

In [67]:
positive_effects(G)

Matrix([
[2, 1, 2, 2, 3, 0],
[0, 1, 1, 1, 0, 1],
[1, 0, 2, 2, 2, 0],
[0, 0, 0, 3, 0, 0],
[2, 0, 3, 3, 4, 1],
[0, 1, 1, 1, 0, 2]])

In [68]:
negative_effects(G)

Matrix([
[0, 1, 0, 0, 1, 2],
[1, 0, 1, 1, 2, 1],
[0, 1, 0, 0, 0, 2],
[0, 0, 0, 1, 0, 0],
[0, 2, 1, 1, 0, 3],
[1, 0, 1, 1, 2, 1]])

In [69]:
absolute_effects(G)

Matrix([
[2, 2, 2, 2, 4, 2],
[1, 1, 2, 2, 2, 2],
[1, 1, 2, 2, 2, 2],
[0, 0, 0, 4, 0, 0],
[2, 2, 4, 4, 4, 4],
[1, 1, 2, 2, 2, 3]])

In [70]:
weighted_effects(G).evalf(2)

Matrix([
[ 1.0,    0, 1.0, 1.0,  0.5, -1.0],
[-1.0,  1.0,   0,   0, -1.0,    0],
[ 1.0, -1.0, 1.0, 1.0,  1.0, -1.0],
[ nan,  nan, nan, 0.5,  nan,  nan],
[ 1.0, -1.0, 0.5, 0.5,  1.0, -0.5],
[-1.0,  1.0,   0,   0, -1.0, 0.33]])

In [71]:
sign_determinacy_effects(G).evalf(2)

Matrix([
[ 1.0,  0.5,  1.0,  1.0, 0.86,  -1.0],
[-1.0,  1.0,  0.5,  0.5, -1.0,   0.5],
[ 1.0, -1.0,  1.0,  1.0,  1.0,  -1.0],
[ nan,  nan,  nan, 0.86,  nan,   nan],
[ 1.0, -1.0, 0.86, 0.86,  1.0, -0.86],
[-1.0,  1.0,  0.5,  0.5, -1.0,  0.77]])

In [72]:
table_of_predictions(sign_determinacy_effects(G), t1=0.8, t2=1, index=get_nodes(G, 'state') + get_nodes(G, 'output'), columns=get_nodes(G, 'state') + get_nodes(G, 'input'))

,1,2,3,4,5,6
1,+,?,+,+,(+),−
2,−,+,?,?,−,?
3,+,−,+,+,+,−
4,0,0,0,(+),0,0
7,+,−,(+),(+),+,(−)
8,−,+,?,?,−,?


In [73]:
simulation_effects(G).evalf(2)

Matrix([
[ 1.0, 0.58,  1.0,  1.0, 0.83,  -1.0],
[-1.0,  1.0, 0.58, 0.58, -1.0, -0.58],
[ 1.0, -1.0,  1.0,  1.0,  1.0,  -1.0],
[ nan,  nan,  nan,  1.0,  nan,   nan],
[ 1.0, -1.0, 0.83, 0.83,  1.0, -0.83],
[-1.0,  1.0, 0.58, 0.58, -1.0,  0.88]])

In [74]:
table_of_predictions(simulation_effects(G), t1=0.8, t2=1, index=get_nodes(G, 'state') + get_nodes(G, 'output'), columns=get_nodes(G, 'state') + get_nodes(G, 'input'))

,1,2,3,4,5,6
1,+,?,+,+,(+),−
2,−,+,?,?,−,?
3,+,−,+,+,+,−
4,0,0,0,+,0,0
7,+,−,(+),(+),+,(−)
8,−,+,?,?,−,(+)


### Informative indicators

In [75]:
mutual_information(G, n_sim=10000, seed=42)

,Node,Mutual Information
0,3,0.918296
1,4,0.650022
2,7,0.615811
3,8,0.584329
4,1,0.560345
5,2,0.494640


### Model validation

In [84]:
perturb = ('3', -1)
observe = (('2', -1), ('7', -1))
marginal_likelihood(G, perturb, observe)

0.4091

In [85]:
model_validation(G, perturb, observe)

,Edges,Model A,Model B,Model C,Model D
0,2 ⊸ 2,,✓,,✓
1,3 ⟶ 1,,,✓,✓
2,────────────────────,────────,────────,────────,────────
3,Marginal likelihood,0.501,0.714,0.276,0.409


In [86]:
predictions = posterior_predictions(G, perturb, observe=None).evalf(2)
predictions

Matrix([
[ -1.0],
[-0.58],
[ -1.0],
[  nan],
[-0.83],
[-0.58]])

In [87]:
matches = posterior_predictions(G, perturb, observe=observe).evalf(2)
matches

Matrix([
[-1.0],
[-1.0],
[-1.0],
[ nan],
[-1.0],
[-1.0]])

In [89]:
diagnose_observations(G, observe)

,Perturbed node,Perturbation sign,Marginal likelihood
0,3,-1,0.4091
1,4,-1,0.4091
2,6,1,0.4091
3,1,1,0.0000
4,1,-1,0.0000
5,2,1,0.0000
6,2,-1,0.0000
7,3,1,0.0000
8,4,1,0.0000
9,5,1,0.0000


## Symbolic analyses

### Structure

In [90]:
create_matrix(G, form='symbolic', matrix_type='A')

Matrix([
[-a_1,1,  a_1,2,  a_1,3,      0],
[-a_2,1, -a_2,2,  a_2,3,      0],
[     0, -a_3,2, -a_3,3,  a_3,4],
[     0,      0,      0, -a_4,4]])

### Stability

In [91]:
system_feedback(G)

Matrix([
[                                                                                                                                                                                -1],
[                                                                                                                                                    -a_1,1 - a_2,2 - a_3,3 - a_4,4],
[                                                                    -a_1,1*a_2,2 - a_1,1*a_3,3 - a_1,1*a_4,4 - a_1,2*a_2,1 - a_2,2*a_3,3 - a_2,2*a_4,4 - a_2,3*a_3,2 - a_3,3*a_4,4],
[-a_1,1*a_2,2*a_3,3 - a_1,1*a_2,2*a_4,4 - a_1,1*a_2,3*a_3,2 - a_1,1*a_3,3*a_4,4 - a_1,2*a_2,1*a_3,3 - a_1,2*a_2,1*a_4,4 + a_1,3*a_2,1*a_3,2 - a_2,2*a_3,3*a_4,4 - a_2,3*a_3,2*a_4,4],
[                                                                            -a_1,1*a_2,2*a_3,3*a_4,4 - a_1,1*a_2,3*a_3,2*a_4,4 - a_1,2*a_2,1*a_3,3*a_4,4 + a_1,3*a_2,1*a_3,2*a_4,4]])

In [92]:
if len(get_nodes(G, 'state')) < 5:
    display(hurwitz_determinants(G))

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [93]:
structural_sensitivity(G)

Matrix([
[-a_1,1*a_2,2*a_3,3*a_4,4 - a_1,1*a_2,3*a_3,2*a_4,4,                           -a_1,2*a_2,1*a_3,3*a_4,4,                            a_1,3*a_2,1*a_3,2*a_4,4,                                                                                                      0],
[-a_1,2*a_2,1*a_3,3*a_4,4 + a_1,3*a_2,1*a_3,2*a_4,4,                           -a_1,1*a_2,2*a_3,3*a_4,4,                           -a_1,1*a_2,3*a_3,2*a_4,4,                                                                                                      0],
[                                                 0, -a_1,1*a_2,3*a_3,2*a_4,4 + a_1,3*a_2,1*a_3,2*a_4,4, -a_1,1*a_2,2*a_3,3*a_4,4 - a_1,2*a_2,1*a_3,3*a_4,4,                                                                                                      0],
[                                                 0,                                                  0,                                                  0, -a_1,1*a_2,2*a_3,3*a_4,4 - a_1,1*a_2,3*a_3,2*a_4

### Press perturbation

In [94]:
adjoint_matrix(G, form='symbolic')

Matrix([
[a_2,2*a_3,3*a_4,4 + a_2,3*a_3,2*a_4,4, a_1,2*a_3,3*a_4,4 - a_1,3*a_3,2*a_4,4, a_1,2*a_2,3*a_4,4 + a_1,3*a_2,2*a_4,4,                                         a_1,2*a_2,3*a_3,4 + a_1,3*a_2,2*a_3,4],
[                   -a_2,1*a_3,3*a_4,4,                     a_1,1*a_3,3*a_4,4, a_1,1*a_2,3*a_4,4 - a_1,3*a_2,1*a_4,4,                                         a_1,1*a_2,3*a_3,4 - a_1,3*a_2,1*a_3,4],
[                    a_2,1*a_3,2*a_4,4,                    -a_1,1*a_3,2*a_4,4, a_1,1*a_2,2*a_4,4 + a_1,2*a_2,1*a_4,4,                                         a_1,1*a_2,2*a_3,4 + a_1,2*a_2,1*a_3,4],
[                                    0,                                     0,                                     0, a_1,1*a_2,2*a_3,3 + a_1,1*a_2,3*a_3,2 + a_1,2*a_2,1*a_3,3 - a_1,3*a_2,1*a_3,2]])

In [95]:
birth_matrix(G, form='symbolic')

Matrix([
[0, a_1,2, a_1,3,     0],
[0,     0, a_2,3,     0],
[0,     0,     0, a_3,4],
[0,     0,     0,     0]])

In [96]:
death_matrix(G, form='symbolic')

Matrix([
[a_1,1,     0,     0,     0],
[a_2,1, a_2,2,     0,     0],
[    0, a_3,2, a_3,3,     0],
[    0,     0,     0, a_4,4]])

In [97]:
life_expectancy_change(G, type='birth')

Matrix([
[-a_1,1*a_2,2*a_3,3*a_4,4 - a_1,1*a_2,3*a_3,2*a_4,4,                           -a_1,1*a_1,2*a_3,3*a_4,4 + a_1,1*a_1,3*a_3,2*a_4,4,                                                     -a_1,1*a_1,2*a_2,3*a_4,4 - a_1,1*a_1,3*a_2,2*a_4,4,                                                     -a_1,1*a_1,2*a_2,3*a_3,4 - a_1,1*a_1,3*a_2,2*a_3,4],
[                          -a_2,1*a_2,3*a_3,2*a_4,4, -a_1,1*a_2,2*a_3,3*a_4,4 - a_1,2*a_2,1*a_3,3*a_4,4 + a_1,3*a_2,1*a_3,2*a_4,4,                                                     -a_1,1*a_2,2*a_2,3*a_4,4 - a_1,2*a_2,1*a_2,3*a_4,4,                                                     -a_1,1*a_2,2*a_2,3*a_3,4 - a_1,2*a_2,1*a_2,3*a_3,4],
[                                                 0,                                                                            0, -a_1,1*a_2,2*a_3,3*a_4,4 - a_1,1*a_2,3*a_3,2*a_4,4 - a_1,2*a_2,1*a_3,3*a_4,4 + a_1,3*a_2,1*a_3,2*a_4,4, -a_1,1*a_2,2*a_3,3*a_3,4 - a_1,1*a_2,3*a_3,2*a_3,4 - a_1,2*a_2,1*a_3,3*a_3,4

In [98]:
life_expectancy_change(G, type='death')

Matrix([
[a_1,2*a_2,1*a_3,3*a_4,4 - a_1,3*a_2,1*a_3,2*a_4,4, -a_1,1*a_1,2*a_3,3*a_4,4 + a_1,1*a_1,3*a_3,2*a_4,4, -a_1,1*a_1,2*a_2,3*a_4,4 - a_1,1*a_1,3*a_2,2*a_4,4,                                                     -a_1,1*a_1,2*a_2,3*a_3,4 - a_1,1*a_1,3*a_2,2*a_3,4],
[                         -a_2,1*a_2,3*a_3,2*a_4,4,                            a_1,1*a_2,3*a_3,2*a_4,4, -a_1,1*a_2,2*a_2,3*a_4,4 - a_1,2*a_2,1*a_2,3*a_4,4,                                                     -a_1,1*a_2,2*a_2,3*a_3,4 - a_1,2*a_2,1*a_2,3*a_3,4],
[                                                0,                                                  0,                                                  0, -a_1,1*a_2,2*a_3,3*a_3,4 - a_1,1*a_2,3*a_3,2*a_3,4 - a_1,2*a_2,1*a_3,3*a_3,4 + a_1,3*a_2,1*a_3,2*a_3,4],
[                                                0,                                                  0,                                                  0,                                                     

In [99]:
perturb='3'

In [100]:
adjoint_matrix(G, form='symbolic', perturb=perturb)

Matrix([
[a_1,2*a_2,3*a_4,4 + a_1,3*a_2,2*a_4,4],
[a_1,1*a_2,3*a_4,4 - a_1,3*a_2,1*a_4,4],
[a_1,1*a_2,2*a_4,4 + a_1,2*a_2,1*a_4,4],
[                                    0]])

In [101]:
life_expectancy_change(G, type='birth', perturb=perturb)

Matrix([
[                                                    -a_1,1*a_1,2*a_2,3*a_4,4 - a_1,1*a_1,3*a_2,2*a_4,4],
[                                                    -a_1,1*a_2,2*a_2,3*a_4,4 - a_1,2*a_2,1*a_2,3*a_4,4],
[-a_1,1*a_2,2*a_3,3*a_4,4 - a_1,1*a_2,3*a_3,2*a_4,4 - a_1,2*a_2,1*a_3,3*a_4,4 + a_1,3*a_2,1*a_3,2*a_4,4],
[                                                                                                     0]])

In [102]:
life_expectancy_change(G, type='death', perturb=perturb)

Matrix([
[-a_1,1*a_1,2*a_2,3*a_4,4 - a_1,1*a_1,3*a_2,2*a_4,4],
[-a_1,1*a_2,2*a_2,3*a_4,4 - a_1,2*a_2,1*a_2,3*a_4,4],
[                                                 0],
[                                                 0]])

### Causal pathways

In [110]:
get_cycles(G)

Matrix([
[           -a_1,1],
[           -a_2,2],
[           -a_3,3],
[           -a_4,4],
[     -a_1,2*a_2,1],
[     -a_2,3*a_3,2],
[a_1,3*a_2,1*a_3,2]])

In [111]:
get_paths(G, source=source, target=target, form='symbolic')

Matrix([
[a_1,2*a_2,3],
[      a_1,3]])

In [112]:
complementary_feedback(G, source=source, target=target, form='symbolic')

Matrix([
[      -a_4,4],
[-a_2,2*a_4,4]])

In [113]:
system_paths(G, source=source, target=target, form='symbolic')

Matrix([
[a_1,2*a_2,3*a_4,4],
[a_1,3*a_2,2*a_4,4]])

### Cumulative effects

In [114]:
create_matrix(G, form='symbolic', matrix_type='A')

Matrix([
[-a_1,1,  a_1,2,  a_1,3,      0],
[-a_2,1, -a_2,2,  a_2,3,      0],
[     0, -a_3,2, -a_3,3,  a_3,4],
[     0,      0,      0, -a_4,4]])

In [115]:
create_matrix(G, form='symbolic', matrix_type='B')

Matrix([
[ b_1,5,      0],
[-b_2,5,      0],
[     0, -b_3,6],
[     0,      0]])

In [116]:
create_matrix(G, form='symbolic', matrix_type='C')

Matrix([
[0, -c_7,2, c_7,3, 0],
[0,  c_8,2,     0, 0]])

In [117]:
create_matrix(G, form='symbolic', matrix_type='D')

Matrix([
[0,     0],
[0, d_8,6]])

In [118]:
create_equations(G, form='state')

Matrix([
[-a_1,1*x_1 + a_1,2*x_2 + a_1,3*x_3 + b_1,5*u_5],
[-a_2,1*x_1 - a_2,2*x_2 + a_2,3*x_3 - b_2,5*u_5],
[-a_3,2*x_2 - a_3,3*x_3 + a_3,4*x_4 - b_3,6*u_6],
[                                    -a_4,4*x_4]])

In [119]:
create_equations(G, form='output')

Matrix([
[-c_7,2*x_2 + c_7,3*x_3],
[ c_8,2*x_2 + d_8,6*u_6]])

In [120]:
cumulative_effects(G)

Matrix([
[            a_2,2*a_3,3*a_4,4 + a_2,3*a_3,2*a_4,4,              a_1,2*a_3,3*a_4,4 - a_1,3*a_3,2*a_4,4,                                                                 a_1,2*a_2,3*a_4,4 + a_1,3*a_2,2*a_4,4,                                                                 a_1,2*a_2,3*a_3,4 + a_1,3*a_2,2*a_3,4,                        -a_1,2*a_3,3*a_4,4*b_2,5 + a_1,3*a_3,2*a_4,4*b_2,5 + a_2,2*a_3,3*a_4,4*b_1,5 + a_2,3*a_3,2*a_4,4*b_1,5,                                                                             -a_1,2*a_2,3*a_4,4*b_3,6 - a_1,3*a_2,2*a_4,4*b_3,6],
[                               -a_2,1*a_3,3*a_4,4,                                  a_1,1*a_3,3*a_4,4,                                                                 a_1,1*a_2,3*a_4,4 - a_1,3*a_2,1*a_4,4,                                                                 a_1,1*a_2,3*a_3,4 - a_1,3*a_2,1*a_3,4,                                                                            -a_1,1*a_3,3*a_4,4*b_2,5 - a_2,1*a_3,3*a